In [1]:
import os
import pandas as pd
from PIL import Image


In [2]:
# Root of your project
PROJECT_ROOT = r"D:/PEC/HSI_Project"

# Raw data path
RAW_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "raw_data",
    "Segregated Sentences and Parts"
)

# Output CSV path
OUTPUT_CSV = os.path.join(
    PROJECT_ROOT,
    "processed",
    "dataset_index.csv"
)

print("Raw data path:", RAW_DATA_PATH)
print("Output CSV will be saved at:", OUTPUT_CSV)


Raw data path: D:/PEC/HSI_Project\raw_data\Segregated Sentences and Parts
Output CSV will be saved at: D:/PEC/HSI_Project\processed\dataset_index.csv


In [3]:
dataset_rows = []

for root, dirs, files in os.walk(RAW_DATA_PATH):

    # Collect band images
    band_files = sorted([
        f for f in files
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))
    ])

    # Only process folders that actually contain hyperspectral bands
    if len(band_files) == 0:
        continue

    # Expect exactly 149 bands
    num_bands = len(band_files)

    # Read one band to get image size
    first_band_path = os.path.join(root, band_files[0])
    try:
        with Image.open(first_band_path) as img:
            width, height = img.size
    except:
        continue

    # Parse folder structure
    parts = root.replace(RAW_DATA_PATH, "").strip(os.sep).split(os.sep)

    # Expected structure:
    # [Page, Document, Image, Pen, Sentence]
    if len(parts) != 5:
        continue

    page, document, image, pen, sentence = parts

    dataset_rows.append({
        "page": page,
        "document": document,
        "image": image,
        "pen": pen,
        "sentence": sentence,
        "num_bands": num_bands,
        "height": height,
        "width": width,
        "path": root
    })


In [4]:
df = pd.DataFrame(dataset_rows)

print("Total samples indexed:", len(df))
print("\nSample rows:")
df.head()


Total samples indexed: 8284

Sample rows:


,page,document,image,pen,sentence,num_bands,height,width,path
0,0. Pages 1 and 2,Document_1,Image_01,Pen_1,Sentence_1,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
1,0. Pages 1 and 2,Document_1,Image_01,Pen_1,Sentence_2,149,78,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
2,0. Pages 1 and 2,Document_1,Image_01,Pen_2,Sentence_1,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
3,0. Pages 1 and 2,Document_1,Image_01,Pen_2,Sentence_2,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
4,0. Pages 1 and 2,Document_1,Image_01,Pen_3,Sentence_1,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...


In [5]:
# Make sure processed folder exists
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# Save CSV
df.to_csv(OUTPUT_CSV, index=False)

print("Saved at:", OUTPUT_CSV)


Saved at: D:/PEC/HSI_Project\processed\dataset_index.csv


In [6]:
# Reload and verify
df_check = pd.read_csv(OUTPUT_CSV)

print("Rows:", df_check.shape[0])
print("Columns:", df_check.shape[1])
print("\nColumns list:")
print(df_check.columns.tolist())

print("\nUnique values check:")
print("Documents:", df_check["document"].nunique())
print("Pens:", df_check["pen"].nunique())
print("Sentences:", df_check["sentence"].unique())
print("Band counts:", df_check["num_bands"].unique())
print("Image sizes:", df_check[["height", "width"]].drop_duplicates().head())


Rows: 8284
Columns: 9

Columns list:
['page', 'document', 'image', 'pen', 'sentence', 'num_bands', 'height', 'width', 'path']

Unique values check:
Documents: 54
Pens: 27
Sentences: ['Sentence_1' 'Sentence_2' 'Combined' 'Pen_1' 'Pen_2' 'Pen_3' 'Pen_4'
 'Pen_5' 'Pen_6' 'Pen_7' 'Pen_8' 'Pen_9' 'Pen_10' 'Pen_11' 'Pen_12'
 'Writer_00' 'Writer_01' 'Writer_02' 'Writer_03' 'Writer_04' 'Writer_05'
 'Writer_06' 'Writer_07' 'Writer_08' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i'
 'j' 'k' 'l' 'm' 'n' 'o' 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z' 'A'
 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'T' 'U' 'V' 'W' 'X' 'Y' 'Z' '0' '1' '145' '2' '236' '3' '4' '5' '6' '7'
 '789' '8' '9' 'Name' 'Signature']
Band counts: [149 133 148]
Image sizes:     height  width
0       77    515
1       78    515
7       76    515
11      69    515
17      71    515


In [7]:
import pandas as pd

MASTER_CSV = r"D:/PEC/HSI_Project/processed/dataset_index.csv"

df = pd.read_csv(MASTER_CSV)
print("Total samples before filtering:", len(df))


Total samples before filtering: 8284


In [8]:
VALID_SENTENCES = ["Sentence_1", "Sentence_2", "Combined"]

df_sentence = df[df["sentence"].isin(VALID_SENTENCES)].copy()

print("Sentence-level samples:", len(df_sentence))
print("\nSentence distribution:")
print(df_sentence["sentence"].value_counts())


Sentence-level samples: 2307

Sentence distribution:
sentence
Combined      1011
Sentence_1     648
Sentence_2     648
Name: count, dtype: int64


In [9]:
print("\nUnique sentence labels:")
print(df_sentence["sentence"].unique())

print("\nBand counts:")
print(df_sentence["num_bands"].value_counts())

print("\nImage sizes (height × width):")
print(df_sentence[["height", "width"]].drop_duplicates().sort_values(["height", "width"]))



Unique sentence labels:
['Sentence_1' 'Sentence_2' 'Combined']

Band counts:
num_bands
149    2307
Name: count, dtype: int64

Image sizes (height × width):
      height  width
7170      42    515
888       45    515
3029      48    515
2078      49    515
8180      50    515
3129      54    515
3175      55    515
2973      56    515
2962      57    515
4446      59    515
3643      60    515
3638      61    515
7279      64    515
2949      66    515
2214      67    515
3032      68    515
11        69    515
35        70    515
17        71    515
953       72    515
21        74    515
73        75    515
7         76    515
0         77    515
1         78    515


In [10]:
OUTPUT_CSV = r"D:/PEC/HSI_Project/processed/sentence_dataset_index.csv"

df_sentence.to_csv(OUTPUT_CSV, index=False)

print("Clean sentence-level index saved to:")
print(OUTPUT_CSV)


Clean sentence-level index saved to:
D:/PEC/HSI_Project/processed/sentence_dataset_index.csv


In [11]:
df_check = pd.read_csv(OUTPUT_CSV)

print("Rows:", df_check.shape[0])
print("Columns:", df_check.shape[1])
print("Sentence labels:", df_check["sentence"].unique())


Rows: 2307
Columns: 9
Sentence labels: ['Sentence_1' 'Sentence_2' 'Combined']


In [12]:
df = pd.read_csv(
    r"D:/PEC/HSI_Project/processed/sentence_dataset_index.csv"
)

print("Band count distribution:")
print(df["num_bands"].value_counts())


Band count distribution:
num_bands
149    2307
Name: count, dtype: int64


In [13]:
print("Height distribution:")
print(df["height"].value_counts().sort_index())


Height distribution:
height
42      1
45      1
48     30
49     22
50      1
54     19
55     26
56    192
57     28
59      4
60      7
61      1
64      1
66     40
67    109
68     51
69      1
70     24
71     80
72      2
74     94
75    355
76    350
77    775
78     93
Name: count, dtype: int64


In [14]:
print("Unique widths:", df["width"].unique())


Unique widths: [515]


In [15]:
summary = {
    "total_sentence_samples": len(df),
    "sentence_types": df["sentence"].unique().tolist(),
    "band_counts": df["num_bands"].value_counts().to_dict(),
    "height_range": [int(df["height"].min()), int(df["height"].max())],
    "width_unique": df["width"].unique().tolist()
}

summary


{'total_sentence_samples': 2307,
 'sentence_types': ['Sentence_1', 'Sentence_2', 'Combined'],
 'band_counts': {149: 2307},
 'height_range': [42, 78],
 'width_unique': [515]}